# 01 · Data exploration

Goal: load one DB1 subject (or synthetic stand-in if the .mat files aren't downloaded yet), check shapes, and sanity-plot the raw signal and label distribution before any preprocessing.

DB1 facts to keep in mind: 27 subjects, 10 channels, 100 Hz, 52 movements + rest. The signal is **already a rectified envelope** from Otto Bock electrodes — not raw EMG.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt

from src.data import load_or_synthesize, has_real_db1

DATA_DIR = ROOT / 'data' / 'raw'
print('Using real DB1:', has_real_db1(DATA_DIR, subject=1))

In [ ]:
rec = load_or_synthesize(subject=1, data_dir=DATA_DIR, seed=0)
print('emg:', rec.emg.shape, rec.emg.dtype)
print('stimulus:', rec.stimulus.shape, 'unique labels:', np.unique(rec.stimulus).size)
print('repetition:', rec.repetition.shape)
print('fs:', rec.fs)

In [ ]:
# Plot the first 30 seconds of all 10 channels stacked.
n_show = min(rec.emg.shape[0], 30 * rec.fs)
t = np.arange(n_show) / rec.fs
fig, ax = plt.subplots(figsize=(12, 6))
for c in range(rec.emg.shape[1]):
    ax.plot(t, rec.emg[:n_show, c] + 0.5 * c, lw=0.5)
ax.set_xlabel('time (s)')
ax.set_ylabel('channels stacked')
ax.set_title(f'Subject {rec.subject} raw signal (first {n_show/rec.fs:.0f}s)')
plt.show()

In [ ]:
# Class distribution. Rest (class 0) dominates by design — every gesture has rest periods around it.
labels, counts = np.unique(rec.stimulus, return_counts=True)
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(labels, counts)
ax.set_xlabel('class id (0 = rest)')
ax.set_ylabel('samples')
ax.set_title('Per-class sample count for subject 1')
plt.show()
print(f'{len(labels)} unique classes')

## Next
Move to `02_preprocessing.ipynb` to rectify, window, and save processed arrays into `data/processed/`.